# Product Funnel Analysis & A/B Hypothesis Prioritization

Анализ **101 500 сессий** и **24 224 пользователей** ride-hailing приложения.

**Цель:** найти точки потерь воронки, проверить ключевые сегментные сигналы и превратить их в приоритетные SMART-гипотезы для A/B-тестов.

> Анализ наблюдательный: найденные связи используются для генерации гипотез, а не как доказательство причинности.

## 1. Данные и инструменты

Одна строка = сессия. Ключевые события:

`app_opened → price_seen → order_made → ride_completed / user_cancelled`

Используются `pandas`, `matplotlib`, `scipy`. Исходный учебный `.xlsx` не публикуется в репозитории.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

pd.set_option('display.max_columns', None)
df = pd.read_excel('new_dataframe.xlsx')

print('shape:', df.shape)
print('unique users:', df['user_id'].nunique())
display(df.head())

## 2. Data quality

Проверяем пропуски, дубли и логику событий. `surge` и `distance` оказываются **структурными пропусками**: они отсутствуют ровно в тех строках, где пользователь не дошёл до просмотра цены.

In [ ]:
quality = pd.DataFrame({
    'missing': df.isna().sum(),
    'unique': df.nunique(dropna=False)
})
display(quality)

print('full duplicates:', df.duplicated().sum())
print('distance NaN ↔ price_seen=0:',
      df['distance'].isna().equals(df['price_seen'].eq(0)))
print('surge NaN ↔ price_seen=0:',
      df['surge'].isna().equals(df['price_seen'].eq(0)))

checks = {
    'order_without_price': ((df.order_made==1)&(df.price_seen==0)).sum(),
    'complete_without_order': ((df.ride_completed==1)&(df.order_made==0)).sum(),
    'cancel_without_order': ((df.user_cancelled==1)&(df.order_made==0)).sum(),
    'complete_and_cancel': ((df.ride_completed==1)&(df.user_cancelled==1)).sum(),
}
display(pd.Series(checks, name='count'))

## 3. Базовые метрики и воронка

Важно различать:

- `user_cancelled.mean()` = доля отмен среди **всех сессий**;
- `user_cancelled / order_made` = post-order cancellation rate.

Для продуктовой воронки второй знаменатель корректнее.

In [ ]:
funnel = pd.Series({
    'App opened': df.app_opened.sum(),
    'Price seen': df.price_seen.sum(),
    'Order made': df.order_made.sum(),
    'Ride completed': df.ride_completed.sum(),
    'User cancelled': df.user_cancelled.sum()
})

metrics = pd.Series({
    'sessions': len(df),
    'users': df.user_id.nunique(),
    'sessions_per_user': len(df)/df.user_id.nunique(),
    'open_to_price': funnel['Price seen']/funnel['App opened'],
    'price_to_order': funnel['Order made']/funnel['Price seen'],
    'order_to_complete': funnel['Ride completed']/funnel['Order made'],
    'order_to_cancel': funnel['User cancelled']/funnel['Order made'],
    'open_to_complete': funnel['Ride completed']/funnel['App opened']
})
display(metrics)

dropoffs = pd.DataFrame({
    'transition': ['Open → Price','Price → Order','Order → Complete'],
    'from': [funnel['App opened'],funnel['Price seen'],funnel['Order made']],
    'to': [funnel['Price seen'],funnel['Order made'],funnel['Ride completed']]
})
dropoffs['lost'] = dropoffs['from'] - dropoffs['to']
dropoffs['loss_rate'] = dropoffs['lost']/dropoffs['from']
display(dropoffs)

ax = funnel[['App opened','Price seen','Order made','Ride completed']].plot(
    kind='bar', legend=False, figsize=(8,4), title='Product funnel'
)
ax.set_ylabel('Sessions')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Ключевой провал:** `Price seen → Order made` = **81,19%**, то есть теряется **17 195 сессий (18,81%)** после просмотра цены. Post-order cancellation rate = **15,18%**.

## 4. Основные продуктовые сигналы

### Surge и часы

Среди сессий с просмотренной ценой:

- без surge: **85,91%** заказов;
- с surge: **72,28%**;
- разница: **13,62 п.п.**

Сильные провалы также видны в **07:00–09:00** и **19:00–21:00**. Они сохраняются даже при сравнении одинакового статуса surge, поэтому временной эффект не сводится только к повышающему коэффициенту.

In [ ]:
price = df[df.price_seen==1]

surge = price.groupby('surge').agg(
    sessions=('user_id','size'),
    order_rate=('order_made','mean')
)
display(surge)

hour_surge = price.groupby(['hour','surge']).order_made.mean().unstack()
display(hour_surge)
hour_surge.plot(figsize=(9,4), marker='o', title='Price → Order by hour and surge')
plt.ylabel('Conversion')
plt.tight_layout()
plt.show()

### Класс поездки, ОС и география

До заказа классы похожи, но после заказа отмены различаются:

- business: **11,60%**
- comfort: **13,21%**
- economy: **17,81%**

Наиболее устойчивый широкий сигнал связан с ОС: Android имеет более низкий `Price → Order` и более высокий cancellation rate. Разрыв сохраняется в разрезах класса, RFM, возраста, дистанции и `city_center_order`.

Заказы вне центра также хуже: cancellation **18,93%** против **12,67%** в центре. В сегменте `Android × outside center × distance ≤ 4,28 км` cancellation достигает **24–25%**.

In [ ]:
segment = (
    df.groupby(['os','city_center_order'])
      .agg(price=('price_seen','sum'),
           orders=('order_made','sum'),
           cancelled=('user_cancelled','sum'))
)
segment['price_to_order'] = segment.orders/segment.price
segment['cancel_rate'] = segment.cancelled/segment.orders
display(segment)

## 5. Статистическая проверка Android/iOS

`os` не является постоянной характеристикой пользователя: большинство `user_id` встречаются на обеих ОС. Поэтому независимый t-test по сессиям был бы методологически слабым.

Проверяем сигнал **парно на уровне пользователя**: для людей, имеющих наблюдения на обеих ОС, считаем пользовательские rates и применяем критерий Уилкоксона.

In [ ]:
paired_order = (
    df[df.price_seen==1]
    .groupby(['user_id','os']).order_made.mean()
    .unstack().dropna(subset=['Android','iOS'])
)
paired_cancel = (
    df[df.order_made==1]
    .groupby(['user_id','os']).user_cancelled.mean()
    .unstack().dropna(subset=['Android','iOS'])
)

order_test = wilcoxon(paired_order.Android, paired_order.iOS)
cancel_test = wilcoxon(paired_cancel.Android, paired_cancel.iOS)

result = pd.DataFrame({
    'metric': ['Price → Order','Order → Cancel'],
    'paired_users': [len(paired_order),len(paired_cancel)],
    'Android': [paired_order.Android.mean(),paired_cancel.Android.mean()],
    'iOS': [paired_order.iOS.mean(),paired_cancel.iOS.mean()],
    'diff_pp': [
        (paired_order.Android.mean()-paired_order.iOS.mean())*100,
        (paired_cancel.Android.mean()-paired_cancel.iOS.mean())*100
    ],
    'p_value': [order_test.pvalue,cancel_test.pvalue]
})
display(result)

Результат парной проверки:

- `Price → Order`: **77,50% Android vs 85,64% iOS**, разница **−8,13 п.п.**, `p ≈ 3,6×10⁻¹³⁸`;
- `Order → Cancel`: **18,90% vs 11,01%**, разница **+7,89 п.п.**, `p ≈ 8,9×10⁻¹¹⁴`.

Это подтверждает устойчивость сигнала, но **не доказывает причинный эффект ОС или интерфейса**.

## 6. Что не объясняет ключевые различия

Дополнительные разрезы показали:

- RFM-структура Android/iOS почти одинакова;
- возрастные распределения почти одинаковы;
- дистанции почти одинаковы;
- surge почти не связан с отменой уже оформленного `economy`;
- активность пользователя (1, 2, 3–4, 5+ заказов) почти не меняет cancellation rate.

Эти проверки уменьшают число простых альтернативных объяснений, но не заменяют многомерную модель или эксперимент.

## 7. SMART-гипотезы

| № | Гипотеза | Источник | Primary metric |
|---|---|---|---|
| 1 | Более понятное объяснение повышенной стоимости при surge → за 2 недели `Price → Order` вырастет с 72,3% до ≥75,3% | surge 72,28% vs no-surge 85,91% | `Price → Order` |
| 2 | Android: сократить экран подтверждения до ключевых параметров и одной CTA → +4 п.п. за 2 недели | устойчивый Android/iOS-разрыв ≈8 п.п. | `Price → Order` |
| 3 | Android: сделать ETA заметнее после заказа → cancellation −3 п.п. | Android отменяет чаще во всех классах | `Cancel / Order` |
| 4 | Вне центра: добавить подтверждение точки подачи → cancellation 18,9% → ≤16,9% | outside 18,93% vs center 12,67% | `Cancel / Order` |
| 5 | Android × outside × ≤4,3 км: подтверждение точки подачи → cancellation −4 п.п. | сегмент 24–25% | `Cancel / Order` |
| 6 | В часы пик сделать ETA заметнее после расчёта → `Price → Order` +3 п.п. | peak 69–72% vs 84–86% | `Price → Order` |
| 7 | В проблемном сегменте в пик ограничить surge → `Price → Order` +5 п.п. | peak+surge 58,42% vs 71,60% без surge | `Price → Order` |

Целевые эффекты — продуктовые ориентиры. Для реального теста MDE должен определяться через бизнес-ценность, трафик и мощность.

## 8. Первичная оценка 1–10

| № | Оценка | Почему | Почему не первой |
|---|---:|---|---|
| 1 | **9** | большой surge-gap, простой тест коммуникации | причина может быть не в тексте |
| 2 | **9** | большой охват, устойчивый OS-сигнал | конкретный UX-дефект ещё не локализован |
| 3 | **8** | большой post-order Android gap | ETA — предполагаемый механизм |
| 4 | **8** | заметный outside-center gap | причина не установлена |
| 5 | **7** | высокий cancellation | узкий и пересекающийся сегмент |
| 6 | **8** | устойчивый peak-сигнал | ETA не наблюдается в данных |
| 7 | **6** | потенциально большой эффект | высокий ценовой и supply-риск |

## 9. ICE-приоритизация

`ICE = Impact × Confidence × Ease`, каждая оценка 1–10.

Для H7 Confidence и Ease снижены: наблюдаемый разрыв велик, но surge не рандомизирован, а вмешательство в цену затрагивает экономику и supply.

In [ ]:
ice = pd.DataFrame({
    'hypothesis':['H1 Surge communication','H2 Android UX','H3 Android ETA',
                  'H4 Outside pickup','H5 Narrow segment','H6 Peak ETA','H7 Surge restriction'],
    'Impact':[8,9,8,8,9,7,10],
    'Confidence':[7,8,6,6,7,6,7],
    'Ease':[7,7,6,7,5,5,4]
})
ice['ICE'] = ice.Impact*ice.Confidence*ice.Ease
ice = ice.sort_values('ICE',ascending=False).reset_index(drop=True)
display(ice)

ax = ice.sort_values('ICE').plot.barh(
    x='hypothesis', y='ICE', legend=False, figsize=(8,4), title='ICE ranking'
)
ax.set_xlabel('ICE Score')
plt.tight_layout()
plt.show()

**TOP-3:** H2 = **504**, H1 = **392**, H4 = **336**.  
H7 после корректировки = **280**: большой наблюдаемый gap не делает ценовой эксперимент первым по приоритету.

## 10. Мини-дизайн A/B для TOP-3

| Hypothesis | Population | Randomization | Primary | Secondary / guardrails |
|---|---|---|---|---|
| **H2 Android UX** | Android, `price_seen=1` | `user_id` | `Price → Order` | completion, cancellation, technical errors |
| **H1 Surge communication** | `price_seen=1`, `surge=surge` | `user_id` | `Price → Order` | completion, cancellation; revenue/margin/ETA if available |
| **H4 Outside pickup** | outside-center eligible users | `user_id` | cancellation | `Price → Order`, completed rides per eligible session |

Рандомизация по `user_id` нужна, чтобы один пользователь не видел оба варианта.

Для H4 нельзя смотреть только на cancellation среди заказавших: изменение происходит **до заказа** и может само изменить состав заказавших. Поэтому одновременно контролируются `Price → Order` и completed rides per eligible session.

## 11. Итог и следующий шаг

Проект проходит цепочку:

**data quality → funnel → segmentation → alternative explanations → paired statistical check → SMART hypotheses → 1–10 scoring → ICE → A/B design**

Следующий этап полноценного эксперимента:

1. выбрать TOP-гипотезу;
2. сформулировать `H0/H1`;
3. определить бизнес-значимый MDE;
4. рассчитать sample size и длительность;
5. заранее зафиксировать правила анализа;
6. провести тест по `user_id`;
7. оценить эффект, доверительный интервал и guardrails.

### Ограничения

- данные наблюдательные;
- большинство сегментных сравнений описательные;
- нет цены, выручки, маржи, ETA, supply и дорожной ситуации;
- последовательный поиск сегментов требует подтверждения на новых данных или в эксперименте.

---
### Воспроизводимость

Файл `new_dataframe.xlsx` должен лежать рядом с ноутбуком.

```bash
pip install pandas matplotlib scipy openpyxl
```